# Fight Week Revenue Playbook - Cleaning Notebook
**Sample (synthetic) data. Fighters, promoters, venues, hotels and sponsors are fictional. DAZN fees and costs are assumptions.**

Upload `fight-week-revenue-playbook.zip` into the Files panel on the left, then run each cell from top to bottom (Shift + Enter).

### Cell 1: Open (unzip) your data

In [ ]:
import zipfile, glob, os
with zipfile.ZipFile("fight-week-revenue-playbook.zip") as z:
    z.extractall(".")
RAW = [p for p in glob.glob("**/data_raw", recursive=True) if "__MACOSX" not in p][0]
print("Data folder:", RAW)
for f in sorted(os.listdir(RAW)):
    print(" ", f)

### Cell 2: Get the tools ready

In [ ]:
import pandas as pd
import numpy as np
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

### Cell 3: Load all 11 files (everything loads as text first)

In [ ]:
def load(name):
    return pd.read_csv(f"{RAW}/{name}", dtype=str, keep_default_na=False)
events, orders, buyers = load("events_raw.csv"), load("ticket_orders_raw.csv"), load("buyers_raw.csv")
bouts, staff, xlines = load("bouts_raw.csv"), load("staff_raw.csv"), load("expense_lines_raw.csv")
acts, hosp, hotels = load("fight_week_activities_raw.csv"), load("hospitality_sales_raw.csv"), load("hotel_blocks_raw.csv")
sponsors, mkt = load("sponsor_deals_raw.csv"), load("marketing_channels_raw.csv")
ALL = {"events": events, "orders": orders, "buyers": buyers, "bouts": bouts, "staff": staff, "expense lines": xlines,
       "activities": acts, "hospitality": hosp, "hotels": hotels, "sponsors": sponsors, "marketing": mkt}
for name, df in ALL.items():
    print(f"{name:14s}", df.shape)
print("total rows:", sum(len(d) for d in ALL.values()))

### Cell 4: Look for problems before fixing anything

In [ ]:
print("Duplicate event rows:   ", events.duplicated().sum())
print("Duplicate Order_IDs:    ", orders.Order_ID.duplicated().sum())
print("Duplicate Buyer_IDs:    ", buyers.Buyer_ID.duplicated().sum())
print("Duplicate bout rows:    ", bouts.duplicated().sum())
print("Expense lines entered twice:", xlines.Line_ID.duplicated().sum())
print("Blank ticket prices:    ", (orders.Price_Each.str.strip() == "").sum())
print("Seat tier spellings:    ", orders.Seat_Tier.nunique())
print("Expense category spellings:", xlines.Category.nunique())
print()
print("Messy money:", events.DAZN_Rights_Fee.head(6).tolist())

### Cell 5: Clean the events and add each card's DAZN fee

In [ ]:
CITY_FIX = {"nyc": "New York", "new york, ny": "New York", "new york": "New York", "brooklyn, ny": "Brooklyn", "bklyn": "Brooklyn",
            "brooklyn": "Brooklyn", "vegas": "Las Vegas", "las vegas": "Las Vegas", "atl": "Atlanta", "atlanta, ga": "Atlanta",
            "atlanta": "Atlanta", "dc": "Washington", "washington dc": "Washington", "washington, d.c.": "Washington",
            "washington": "Washington", "national harbor": "Oxon Hill", "oxon hill, md": "Oxon Hill", "oxon hill": "Oxon Hill",
            "philly": "Philadelphia", "phl": "Philadelphia", "philadelphia, pa": "Philadelphia", "philadelphia": "Philadelphia",
            "htx": "Houston", "houston, tx": "Houston", "h-town": "Houston", "houston": "Houston",
            "sa": "San Antonio", "san antonio, tx": "San Antonio", "san antonio": "San Antonio"}
CITY_TO_MARKET = {"New York": "New York", "Brooklyn": "New York", "Las Vegas": "Las Vegas", "Atlanta": "Atlanta",
                  "Washington": "DMV", "Oxon Hill": "DMV", "Philadelphia": "Philadelphia", "Houston": "Texas", "San Antonio": "Texas"}
MID = ["DMV", "Philadelphia", "Texas"]

def to_money(s):
    """Turns $1.85M, $250K, 1,234 or $1,234.56 into a number."""
    s = s.astype(str).str.strip().str.upper().str.replace(r"[$,\s]|USD", "", regex=True)
    mult = np.where(s.str.endswith("M"), 1e6, np.where(s.str.endswith("K"), 1e3, 1))
    num = pd.to_numeric(s.str.rstrip("MK").replace("", np.nan), errors="coerce")
    return num * mult

def is_yes(s):
    return s.str.strip().str.lower().isin(["yes", "y", "true", "1"])

ev = events.drop_duplicates().copy()
ev["Event_Date"] = pd.to_datetime(ev.Event_Date, format="mixed")
ev["City"] = ev.City.str.strip().str.lower().map(CITY_FIX)
ev["Market"] = ev.City.map(CITY_TO_MARKET)
ev["Card_Type"] = np.where(ev.Market.isin(MID), "Mid-sized", "Big")
ev["Venue_Capacity"] = to_money(ev.Venue_Capacity).astype(int)
ev["Tickets_Sold"] = to_money(ev.Tickets_Sold).astype(int)
ev["DAZN_Rights_Fee"] = to_money(ev.DAZN_Rights_Fee)
ev["Team_Hotel_Rate"] = ev.Team_Hotel_Rate.astype(float)
ev["Headliner_Tier"] = ev.Headliner_Tier.str.strip().str.title()
ev["Headliner_Is_Local"] = is_yes(ev.Headliner_Is_Local)
ev["Latitude"], ev["Longitude"] = ev.Latitude.astype(float), ev.Longitude.astype(float)
ev["Room_Size"] = np.where(ev.Venue_Capacity >= 10000, "Arena (15K+)", "Mid-sized room (4K-7.5K)")
ev["Sell_Through"] = (ev.Tickets_Sold / ev.Venue_Capacity).round(3)
print("events:", len(ev), "| missing markets:", ev.Market.isna().sum(), "| missing DAZN fees:", ev.DAZN_Rights_Fee.isna().sum())
print(ev.groupby(["Card_Type", "Market"]).agg(Cards=("Event_ID", "count"), Avg_Sell_Through=("Sell_Through", "mean"),
      Avg_DAZN_Fee=("DAZN_Rights_Fee", "mean")).round(3))

### Cell 6: Remove duplicate orders and fix the dates

In [ ]:
before = len(orders)
od = orders.drop_duplicates(subset="Order_ID").copy()
print("removed", before - len(od))
od["Order_Date"] = pd.to_datetime(od.Order_Date, format="mixed")
od["Tickets"] = od.Tickets.astype(int)
print("earliest:", od.Order_Date.min().date(), "| latest:", od.Order_Date.max().date())

### Cell 7: Fix seat names and sales channels

In [ ]:
def squash(s):
    return s.str.strip().str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
SEAT_FIX = {"ringside": "Ringside", "vip": "VIP", "floor": "Floor", "lowerbowl": "Lower Bowl", "lower": "Lower Bowl", "lb": "Lower Bowl",
            "upperbowl": "Upper Bowl", "upper": "Upper Bowl", "ub": "Upper Bowl"}
CHANNEL_FIX = {"presaleemail": "Presale Email", "presale": "Presale Email", "emailpresale": "Presale Email", "ticketmaster": "Ticketmaster",
               "tm": "Ticketmaster", "online": "Ticketmaster", "boxoffice": "Box Office", "gympartner": "Gym Partner", "gym": "Gym Partner",
               "barbershoppartner": "Barbershop Partner", "barbershop": "Barbershop Partner", "fighterlink": "Fighter Link",
               "fighterslink": "Fighter Link", "promocode": "Promo Code", "promo": "Promo Code"}
od["Seat_Tier"] = squash(od.Seat_Tier).map(SEAT_FIX)
od["Sales_Channel"] = squash(od.Sales_Channel).map(CHANNEL_FIX)
print(od.Seat_Tier.value_counts(dropna=False), "\n")
print(od.Sales_Channel.value_counts(dropna=False))

### Cell 8: Turn prices into numbers, flag free (comp) tickets, fill the few blank prices

In [ ]:
raw = od.Price_Each.str.strip()
od["Is_Comp"] = raw.str.lower().isin(["comp", "0", "$0.00"])
od["Price_Each"] = to_money(raw)
od.loc[od.Is_Comp, "Price_Each"] = 0.0
blank = od.Price_Each.isna()
print("blank prices:", blank.sum())
typical = od[~od.Is_Comp].groupby(["Event_ID", "Seat_Tier"]).Price_Each.median()
od["Price_Was_Filled"] = blank
od.loc[blank, "Price_Each"] = [typical.get((e, t)) for e, t in zip(od.loc[blank, "Event_ID"], od.loc[blank, "Seat_Tier"])]
od["Order_Revenue"] = (od.Tickets * od.Price_Each).round(2)
print("comps:", od.Is_Comp.sum(), "| still blank:", od.Price_Each.isna().sum())

### Cell 9: Add fight info and count how many days before the fight people bought

In [ ]:
od = od.merge(ev[["Event_ID", "Event_Date", "City", "Market", "Card_Type", "Venue", "Venue_Capacity", "Latitude", "Longitude",
                  "Headliner_Tier", "Headliner_Is_Local", "Room_Size"]], on="Event_ID", how="left")
od["Days_Before_Fight"] = (od.Event_Date - od.Order_Date).dt.days
od["Buy_Window"] = pd.cut(od.Days_Before_Fight, [-1, 7, 30, 60, 999], labels=["Final week", "8-30 days", "31-60 days", "60+ days"]).astype(str)
print(od.Buy_Window.value_counts(normalize=True).round(3))

### Cell 10: Clean the buyers and measure how far each one traveled

In [ ]:
HOME_FIX = {"nyc": "New York", "new york, ny": "New York", "atl": "Atlanta", "dc": "Washington", "washington, d.c.": "Washington",
            "philly": "Philadelphia", "htx": "Houston", "vegas": "Las Vegas", "sa": "San Antonio"}
HOME_COORDS = {"New York": (40.71, -74.01), "Las Vegas": (36.17, -115.14), "Atlanta": (33.75, -84.39), "Washington": (38.90, -77.04),
               "Baltimore": (39.29, -76.61), "Philadelphia": (39.95, -75.17), "Wilmington": (39.74, -75.55), "Camden": (39.93, -75.12),
               "Houston": (29.76, -95.37), "San Antonio": (29.42, -98.49), "Austin": (30.27, -97.74), "Dallas": (32.78, -96.80),
               "Charlotte": (35.23, -80.84), "Birmingham": (33.52, -86.80), "Miami": (25.76, -80.19), "Richmond": (37.54, -77.44),
               "Newark": (40.74, -74.17), "Los Angeles": (34.05, -118.24), "Phoenix": (33.45, -112.07), "Chicago": (41.88, -87.63),
               "Detroit": (42.33, -83.05)}
AGE_FIX = {"1824": "18-24", "2534": "25-34", "3544": "35-44", "4554": "45-54", "55": "55+"}
before = len(buyers)
bu = buyers.drop_duplicates(subset="Buyer_ID").copy()
print("removed", before - len(bu))
c = bu.Home_City.str.strip()
bu["Home_City"] = c.str.lower().map(HOME_FIX).fillna(c.str.title()).replace("", "Unknown")
bu["Age_Group"] = bu.Age_Group.str.replace(r"[^0-9]", "", regex=True).map(AGE_FIX)
bu["Email_Opt_In"] = bu.Email_Opt_In.str.strip().str.lower().isin(["yes", "y", "true"])
print("blank home cities:", (bu.Home_City == "Unknown").sum())

def miles(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 3958.8 * 2 * np.arcsin(np.sqrt(a))
od = od.merge(bu[["Buyer_ID", "Home_City", "Age_Group", "Email_Opt_In"]], on="Buyer_ID", how="left")
home = od.Home_City.map(HOME_COORDS)
od["Home_Lat"] = home.map(lambda x: x[0] if isinstance(x, tuple) else np.nan)
od["Home_Lon"] = home.map(lambda x: x[1] if isinstance(x, tuple) else np.nan)
od["Miles_Traveled"] = miles(od.Home_Lat, od.Home_Lon, od.Latitude, od.Longitude).round(0)
od["Fan_Type"] = np.select([od.Miles_Traveled.isna(), od.Miles_Traveled >= 75], ["Unknown", "Traveler"], "Local")
print(od.Fan_Type.value_counts(normalize=True).round(3))

### Cell 11: Count repeat buyers and save the clean ticket file

In [ ]:
od["Buyer_Fights"] = od.Buyer_ID.map(od.groupby("Buyer_ID").Event_ID.nunique())
od["Buyer_Type"] = np.select([od.Buyer_Fights >= 3, od.Buyer_Fights == 2], ["Superfan (3+ fights)", "Repeat (2 fights)"], "One-time")
os.makedirs("data_clean", exist_ok=True)
od.drop(columns=["Home_Lat", "Home_Lon"]).to_csv("data_clean/ticket_orders_clean.csv", index=False)
print("tickets saved:", len(od))

### Cell 12: Clean the fight-week activities

In [ ]:
DAY_FIX = {"wednesday": "1 Wednesday", "wed": "1 Wednesday", "day1": "1 Wednesday", "thursday": "2 Thursday", "thu": "2 Thursday",
           "thurs": "2 Thursday", "day2": "2 Thursday", "friday": "3 Friday", "fri": "3 Friday", "day3": "3 Friday",
           "saturday": "4 Saturday", "sat": "4 Saturday", "fightnight": "4 Saturday", "day4": "4 Saturday"}
ACT_FIX = {"pressconference": "Press Conference", "presser": "Press Conference", "pressconf": "Press Conference",
           "finalpressconference": "Press Conference", "openworkout": "Open Workout", "openworkouts": "Open Workout",
           "mediaworkout": "Open Workout", "weighin": "Weigh-In", "weighins": "Weigh-In", "ceremonialweighin": "Weigh-In",
           "fanfest": "Fan Fest", "fanfestival": "Fan Fest", "afterparty": "After Party", "officialafterparty": "After Party",
           "vipmeetgreet": "VIP Meet & Greet", "meetandgreet": "VIP Meet & Greet", "vipmg": "VIP Meet & Greet"}
fw = acts.copy()
fw["Day"] = squash(fw.Day).map(DAY_FIX)
fw["Activity"] = squash(fw.Activity).map(ACT_FIX)
fw["Ticketed"] = is_yes(fw.Ticketed)
fw["Ticket_Price"] = to_money(fw.Ticket_Price.str.replace("Free", "0")).fillna(0)
fw["Attendance"] = to_money(fw.Attendance)      # blanks stay blank: missing is not zero
fw["Sponsor_On_Site"] = fw.Sponsor_On_Site.str.strip().str.lower().isin(["yes", "y"])
fw["Merch_Revenue"] = fw.Merch_Revenue.astype(float)
fw["Ticket_Revenue"] = (fw.Ticket_Price * fw.Attendance).round(2)
fw["Total_Revenue"] = fw.Ticket_Revenue.fillna(0) + fw.Merch_Revenue
fw = fw.merge(ev[["Event_ID", "Market", "Card_Type"]], on="Event_ID", how="left")
fw.to_csv("data_clean/fight_week_activities_clean.csv", index=False)
print("unmatched days:", fw.Day.isna().sum(), "| unmatched activities:", fw.Activity.isna().sum(), "| blank attendance:", fw.Attendance.isna().sum())

### Cell 13: Clean hospitality packages, fan hotel blocks and sponsor deals

In [ ]:
PKG_FIX = {"ringsideexperience": "Ringside Experience", "ringsideexp": "Ringside Experience", "viplounge": "VIP Lounge",
           "walkoutaccess": "Walkout Access", "walkout": "Walkout Access", "hotelticketbundle": "Hotel + Ticket Bundle",
           "hotelticket": "Hotel + Ticket Bundle", "hotelbundle": "Hotel + Ticket Bundle"}
hp = hosp.copy()
hp["Package"] = squash(hp.Package).map(PKG_FIX)
hp["Price"] = to_money(hp.Price)
hp["Units_Available"], hp["Units_Sold"] = hp.Units_Available.astype(int), hp.Units_Sold.astype(int)
hp["Package_Sell_Through"] = (hp.Units_Sold / hp.Units_Available).round(3)
hp["Sold_Out"] = hp.Package_Sell_Through >= 0.95
hp["Package_Revenue"] = hp.Price * hp.Units_Sold
hp = hp.merge(ev[["Event_ID", "Market", "Card_Type"]], on="Event_ID", how="left")
hp.to_csv("data_clean/hospitality_clean.csv", index=False)

hb = hotels.copy()
hb["Nightly_Rate"] = to_money(hb.Nightly_Rate)
pct = to_money(hb.Commission_Pct.str.replace("%", ""))
hb["Commission_Pct"] = np.where(pct > 1, pct / 100, pct)
for col in ["Rooms_Blocked", "Rooms_Picked_Up", "Nights"]:
    hb[col] = hb[col].astype(int)
hb["Pickup_Rate"] = (hb.Rooms_Picked_Up / hb.Rooms_Blocked).round(3)
hb["Commission_Earned"] = (hb.Rooms_Picked_Up * hb.Nights * hb.Nightly_Rate * hb.Commission_Pct).round(2)
hb = hb.merge(ev[["Event_ID", "Market", "Card_Type"]], on="Event_ID", how="left")
hb.to_csv("data_clean/hotel_blocks_clean.csv", index=False)

sp = sponsors.copy()
sp["Fee"] = to_money(sp.Fee)
sp["Placement"] = np.where(squash(sp.Placement).str.contains("week"), "Fight Week + Fight Night", "Fight Night Only")
sp["Category"] = sp.Category.str.strip().str.title()
sp = sp.merge(ev[["Event_ID", "Market", "Card_Type"]], on="Event_ID", how="left")
sp.to_csv("data_clean/sponsor_deals_clean.csv", index=False)
print("packages:", hp.Package.value_counts().to_dict())
print("commission range:", hb.Commission_Pct.min(), "to", hb.Commission_Pct.max())
print("placements:", sp.Placement.value_counts().to_dict())

### Cell 14: Clean the marketing file (you'll also do this one by hand in Google Sheets)

In [ ]:
MKT_FIX = {"nyc": "New York", "new york, ny": "New York", "vegas": "Las Vegas", "atl": "Atlanta", "atlanta, ga": "Atlanta",
           "dc": "DMV", "washington dc": "DMV", "d.m.v.": "DMV", "philly": "Philadelphia", "phl": "Philadelphia",
           "tx": "Texas", "houston": "Texas"}
mk = mkt.copy()
m = mk.Market.str.strip()
mk["Market"] = m.str.lower().map(MKT_FIX).fillna(m)
mk["Card_Type"] = np.where(mk.Market.isin(MID), "Mid-sized", "Big")
mk["Spend"] = to_money(mk.Spend)
mk["Tickets_Attributed"] = to_money(mk.Tickets_Attributed)
mk["Cost_Per_Ticket"] = (mk.Spend / mk.Tickets_Attributed).round(2)
mk.to_csv("data_clean/marketing_channels_clean.csv", index=False)
print("markets:", sorted(mk.Market.unique()), "| blank ticket counts:", mk.Tickets_Attributed.isna().sum())

### Cell 15: Clean the bouts and price out fighter travel (your fight-week rules)

In [ ]:
# Bella's rules: main event + co-main fighters bring 3 corner people and arrive Tuesday for the press conference (5 hotel nights).
# Everyone else brings 2 corner people and arrives Thursday for the weigh-in (3 nights). Everyone gets $50 a day.
# Assumptions: $450 round-trip flight, 2 people per room, local fighters need no flight or hotel.
FLIGHT, PER_DIEM = 450, 50
bt = bouts.drop_duplicates().copy()
print("removed", len(bouts) - len(bt), "duplicate bouts | bouts:", len(bt))
bt["Bout_Order"] = bt.Bout_Order.astype(int)
bt["Scheduled_Rounds"] = bt.Scheduled_Rounds.str.extract(r"(\d+)")[0].astype(int)
bt["Weight_Class"] = bt.Weight_Class.str.strip().str.title().str.replace("Jr. ", "Super ", regex=False)
bt["Purse_A"], bt["Purse_B"] = to_money(bt.Purse_A), to_money(bt.Purse_B)
bt["A_Is_Local"], bt["B_Is_Local"] = is_yes(bt.A_Is_Local), is_yes(bt.B_Is_Local)
bt["Total_Purse"] = bt.Purse_A + bt.Purse_B
bt["Arrives"] = np.where(bt.Bout_Order <= 2, "Tuesday", "Thursday")
bt["Corner_Size"] = np.where(bt.Bout_Order <= 2, 3, 2)
nights = np.where(bt.Arrives == "Tuesday", 5, 3)
days = nights + 1
bt = bt.merge(ev[["Event_ID", "Team_Hotel_Rate", "Market", "Card_Type"]], on="Event_ID", how="left")
people_per_team = 1 + bt.Corner_Size
travelers = people_per_team * ((~bt.A_Is_Local).astype(int) + (~bt.B_Is_Local).astype(int))
rooms = np.ceil(people_per_team / 2) * ((~bt.A_Is_Local).astype(int) + (~bt.B_Is_Local).astype(int))
bt["People_Traveling"] = travelers
bt["Flights"] = travelers * FLIGHT
bt["Hotel"] = rooms * nights * bt.Team_Hotel_Rate
bt["Per_Diem"] = travelers * days * PER_DIEM
bt["Travel_Cost"] = bt.Flights + bt.Hotel + bt.Per_Diem
bt.to_csv("data_clean/bouts_clean.csv", index=False)
print("weight classes:", bt.Weight_Class.nunique(), "| missing purses:", bt[["Purse_A", "Purse_B"]].isna().sum().sum())
print(bt.groupby("Card_Type")[["Total_Purse", "Travel_Cost"]].sum().div(bt.groupby("Card_Type").Event_ID.nunique(), axis=0).round(0))

### Cell 16: Clean the staff and price out staff costs

In [ ]:
# Bella's rules:
# - Promoter staff: paid $2,500-$5,000. About 10 arrive Tuesday (night before the press conference, 5 nights),
#   the rest arrive Thursday (night before the weigh-in, 3 nights). Flight + own room + $50 a day.
# - Production company staff: the promoter pays the production company a flat fee (in the expense lines),
#   and the company pays its own people. The promoter still covers their hotel and $50 a day. They arrive Thursday.
st = staff.copy()
st["Staff_Type"] = np.where(squash(st.Staff_Type).str.startswith("prod"), "Production Company", "Promoter")
st["Arrival_Day"] = np.where(squash(st.Arrival_Day).str.startswith("tu"), "Tuesday", "Thursday")
st["Fee"] = to_money(st.Fee.str.replace(r"(?i)included in flat fee|n/a|incl\.", "", regex=True)).fillna(0)
st["Is_Local"] = is_yes(st.Is_Local)
st["Role"] = st.Role.str.strip().str.title().str.replace("Pr & Media", "PR & Media", regex=False)
st = st.merge(ev[["Event_ID", "Team_Hotel_Rate", "Card_Type"]], on="Event_ID", how="left")
s_nights = np.where(st.Arrival_Day == "Tuesday", 5, 3)
away = (~st.Is_Local).astype(int)
flight = np.where(st.Staff_Type == "Promoter", FLIGHT, 0)
st["Travel_Cost"] = away * (flight + s_nights * st.Team_Hotel_Rate + (s_nights + 1) * PER_DIEM)
st["Total_Cost"] = st.Fee + st.Travel_Cost
st.to_csv("data_clean/staff_clean.csv", index=False)
crew = st.pivot_table(index="Event_ID", columns="Staff_Type", values="Staff_ID", aggfunc="count")
print("promoter staff fee range:", st[st.Staff_Type == "Promoter"].Fee.min(), "to", st[st.Staff_Type == "Promoter"].Fee.max())
print("promoter staff per card:", crew.Promoter.min(), "to", crew.Promoter.max(),
      "| production staff per card:", crew["Production Company"].min(), "to", crew["Production Company"].max())
print("early arrivals per card:", st[st.Arrival_Day == "Tuesday"].groupby("Event_ID").size().unique())
print(st.groupby(["Card_Type", "Staff_Type"]).Total_Cost.sum().div(st.groupby("Card_Type").Event_ID.nunique(), level=0).round(0))

### Cell 17: Clean the other expense lines (venue, production, commission, officials, medicals)

In [ ]:
CAT_FIX = {"venuerent": "Venue Rent", "venuerental": "Venue Rent", "rentvenue": "Venue Rent",
           "venuesettlement": "Venue Settlement", "venuelaborsettlement": "Venue Settlement", "settlement": "Venue Settlement",
           "productioncompanyflatfee": "Production Company (flat fee)", "productioncoflatfee": "Production Company (flat fee)",
           "productioncompany": "Production Company (flat fee)", "prodcoflat": "Production Company (flat fee)",
           "commissiongatetax": "Commission & Gate Tax", "gatetax": "Commission & Gate Tax", "commissionfee": "Commission & Gate Tax",
           "officialsdoctorsambulance": "Officials, Doctors & Ambulance", "officialsmedicalstaff": "Officials, Doctors & Ambulance",
           "officialsambulance": "Officials, Doctors & Ambulance", "medicaltestinginsurance": "Medical Testing & Insurance",
           "medicalsinsurance": "Medical Testing & Insurance", "fighterinsurancemedicals": "Medical Testing & Insurance"}
xl = xlines.drop_duplicates(subset="Line_ID").copy()
print("removed", len(xlines) - len(xl), "invoices entered twice")
xl["Category"] = squash(xl.Category).map(CAT_FIX)
xl["Amount"] = to_money(xl.Amount)
xl.to_csv("data_clean/expense_lines_clean.csv", index=False)
print("unmatched categories:", xl.Category.isna().sum(), "| blank amounts:", xl.Amount.isna().sum())
print(sorted(xl.Category.unique()))

### Cell 18: Build the profit and loss (P&L) for every card

In [ ]:
SAMPLE = 10   # the ticket file is a 10% sample
by = lambda df, col: df.groupby("Event_ID")[col].sum()
pnl = ev[["Event_ID", "Event_Date", "Market", "Card_Type", "Venue", "Venue_Capacity", "Room_Size", "Headliner_Tier",
          "Headliner_Is_Local", "Tickets_Sold", "Sell_Through", "Latitude", "Longitude"]].set_index("Event_ID")
# money in
pnl["Gate"] = by(od, "Order_Revenue") * SAMPLE
pnl["DAZN_Fee"] = ev.set_index("Event_ID").DAZN_Rights_Fee
pnl["Sponsorship"] = by(sp, "Fee")
pnl["Hospitality"] = by(hp, "Package_Revenue")
pnl["Fight_Week"] = by(fw, "Total_Revenue")
pnl["Hotel_Commission"] = by(hb, "Commission_Earned")
# money out
x = xl.pivot_table(index="Event_ID", columns="Category", values="Amount", aggfunc="sum")
pnl["Purses"] = by(bt, "Total_Purse")
pnl["Fighter_Travel"] = by(bt, "Travel_Cost")
pnl["Staff"] = by(st, "Total_Cost")
pnl["Venue"] = x["Venue Rent"] + x["Venue Settlement"]
pnl["Production"] = x["Production Company (flat fee)"]
pnl["Gate_Tax"] = x["Commission & Gate Tax"]
pnl["Officials_and_Medical"] = x["Officials, Doctors & Ambulance"] + x["Medical Testing & Insurance"]
pnl["Marketing"] = by(mk, "Spend")
pnl = pnl.fillna({c: 0 for c in ["Sponsorship", "Hospitality", "Fight_Week", "Hotel_Commission"]})
IN = ["Gate", "DAZN_Fee", "Sponsorship", "Hospitality", "Fight_Week", "Hotel_Commission"]
OUT = ["Purses", "Fighter_Travel", "Staff", "Venue", "Production", "Gate_Tax", "Officials_and_Medical", "Marketing"]
pnl["Revenue"] = pnl[IN].sum(axis=1)
pnl["Costs"] = pnl[OUT].sum(axis=1)
pnl["Profit"] = pnl.Revenue - pnl.Costs
pnl["Profit_Without_DAZN"] = pnl.Profit - pnl.DAZN_Fee
# break-even: what % of seats must sell to cover everything (gate tax scales with tickets; the rest doesn't)
per_ticket = pnl.Gate / pnl.Tickets_Sold * (1 - pnl.Gate_Tax / pnl.Gate)
fixed = pnl.Costs - pnl.Gate_Tax - (pnl.Revenue - pnl.Gate)
pnl["Break_Even_Sell_Through"] = ((fixed / per_ticket) / pnl.Venue_Capacity).round(3)
pnl["Break_Even_Without_DAZN"] = (((fixed + pnl.DAZN_Fee) / per_ticket) / pnl.Venue_Capacity).round(3)
pnl["Cushion"] = (pnl.Sell_Through - pnl.Break_Even_Sell_Through).round(3)
pnl = pnl.reset_index()
pnl.to_csv("data_clean/event_pnl.csv", index=False)
long = pnl.melt(id_vars=["Event_ID", "Market", "Card_Type", "Room_Size"], value_vars=IN + OUT, var_name="Line", value_name="Amount")
long["Direction"] = np.where(long.Line.isin(IN), "Revenue", "Cost")
long.to_csv("data_clean/pnl_lines.csv", index=False)
print("cards:", len(pnl), "| losing cards:", (pnl.Profit < 0).sum(), "| losing without DAZN:", (pnl.Profit_Without_DAZN < 0).sum())
print(pnl.groupby("Card_Type")[["Revenue", "Costs", "Profit", "Profit_Without_DAZN", "Break_Even_Sell_Through", "Sell_Through"]].mean().round(3))

### Cell 19: Build the market scorecard (where should the next card go?)

In [ ]:
prem = od.assign(P=od.Seat_Tier.isin(["Ringside", "VIP"])).groupby("Market")
score = pnl.groupby("Market").agg(Card_Type=("Card_Type", "first"), Cards=("Event_ID", "count"), Avg_Sell_Through=("Sell_Through", "mean"),
                                  Avg_Profit=("Profit", "mean"), Avg_Cushion=("Cushion", "mean"))
score["Premium_Revenue_Share"] = prem.apply(lambda g: g.loc[g.P, "Order_Revenue"].sum() / g.Order_Revenue.sum())
score["Traveler_Share"] = od[od.Fan_Type != "Unknown"].groupby("Market").Fan_Type.apply(lambda s: (s == "Traveler").mean())
score["Hospitality_Sell_Through"] = hp.groupby("Market").apply(lambda g: g.Units_Sold.sum() / g.Units_Available.sum())
def scale(s): return (s - s.min()) / (s.max() - s.min())
score["Demand_Score"] = (scale(score.Avg_Sell_Through) + scale(score.Premium_Revenue_Share) + scale(score.Traveler_Share)
                         + scale(score.Hospitality_Sell_Through)) / 4 * 100
score["Opportunity_Score"] = (score.Demand_Score * (1 - scale(score.Cards) * 0.5)).round(1)
score = score.sort_values("Opportunity_Score", ascending=False)
score.reset_index().to_csv("data_clean/market_scorecard.csv", index=False)
print(score[["Card_Type", "Cards", "Avg_Sell_Through", "Avg_Profit", "Premium_Revenue_Share", "Opportunity_Score"]].round(3))

### Cell 20: The mid-sized market formula, fight week, buy timing and marketing

In [ ]:
mid = pnl[pnl.Card_Type == "Mid-sized"]
print("MID-SIZED CARDS: room and hometown headliner")
print(mid.groupby(["Room_Size", "Headliner_Is_Local"]).agg(Cards=("Event_ID", "count"), Sell_Through=("Sell_Through", "mean"),
      Break_Even=("Break_Even_Sell_Through", "mean"), Profit=("Profit", "mean")).round(3))
print()
print("FIGHT WEEK (free vs. ticketed)")
print(fw.groupby("Ticketed").agg(Activities=("Activity_ID", "count"), Fans=("Attendance", "sum"), Revenue=("Total_Revenue", "sum")).round(0))
print()
t = od.groupby(["Card_Type", "Buy_Window"]).Tickets.sum()
print("BUY TIMING (share of tickets)")
print((t / t.groupby(level=0).transform("sum")).unstack().round(3))
print()
print("MARKETING: cost per ticket")
print(mk.groupby(["Card_Type", "Channel"]).apply(lambda g: g.Spend.sum() / g.Tickets_Attributed.sum()).unstack(0).round(2).sort_values("Mid-sized"))

### Cell 21: Save the events and buyers

In [ ]:
ev.to_csv("data_clean/events_clean.csv", index=False)
bu.to_csv("data_clean/buyers_clean.csv", index=False)
print(len(os.listdir("data_clean")), "files:", sorted(os.listdir("data_clean")))

### Cell 22: Final test (every answer should say True)

In [ ]:
print(od.Order_ID.is_unique,
      od.Seat_Tier.notna().all() and od.Sales_Channel.notna().all() and od.Price_Each.notna().all(),
      ev.Market.notna().all() and ev.DAZN_Rights_Fee.notna().all(),
      bt[["Purse_A", "Purse_B"]].notna().all().all() and st.Fee.notna().all() and st.Staff_Type.notna().all(),
      xl.Category.notna().all() and xl.Line_ID.is_unique,
      pnl[IN + OUT].notna().all().all(),
      len(os.listdir("data_clean")) == 14)

### Cell 23: Download your clean files (a file called data_clean.zip)

In [ ]:
import shutil
shutil.make_archive("data_clean", "zip", "data_clean")
try:
    from google.colab import files
    files.download("data_clean.zip")
except ImportError:
    print("Not in Colab: data_clean.zip is saved next to this notebook.")